In [2]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 39.1 MB/s eta 0:00:00


In [3]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 17.3 MB/s eta 0:00:00


In [4]:
!pip install evaluate rouge_score

In [5]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch
import evaluate
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [6]:
device = "cuda" if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [10]:
!pip install huggingface_hub
from huggingface_hub import notebook_login

notebook_login()

In [13]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import os

model_name = "google/pegasus-cnn_dailymail" # Corrected model name
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_name, force_download=True).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name, force_download=True)

config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

In [15]:
dataset_samsum = load_dataset("knkarthick/samsum")
dataset_samsum

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [16]:
split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]

print(f"Train set length: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue:")

print(dataset_samsum['test'][0]['dialogue'])

print("\nSummary:")

print(dataset_samsum['test'][0]['summary'])



Train set length: [14731, 818, 819]
Features: ['id', 'dialogue', 'summary']

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


In [18]:
def convert_example_to_features(example_batch):
    # Inputs (Dialogues) ko encode karein
    inputs_encoding = tokenizer(example_batch["dialogue"], truncation=True, padding="max_length", max_length=1024)

    # Targets (Summaries) ko encode karein (Naya Tarika)
    target_encoding = tokenizer(text_target=example_batch["summary"], truncation=True, padding="max_length", max_length=128)

    return {
        "input_ids": inputs_encoding["input_ids"],
        "attention_mask": inputs_encoding["attention_mask"],
        "labels": target_encoding["input_ids"],
    }

In [19]:
dataset_samsum_pt = dataset_samsum.map(convert_example_to_features, batched=True)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [20]:
#Training process
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [23]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
trainer_args = Seq2SeqTrainingArguments(
    output_dir='pegasus-samsum',
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1, # GPU memory ke liye 1 rakhein
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps', # Naya naming convention
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16,
    learning_rate =5e-5,
    fp16=False, # Colab GPU par training fast karne ke liye
    max_grad_norm =1.0
)


In [25]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
trainer = Seq2SeqTrainer(
    model=model_pegasus, args=trainer_args,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["test"],
    eval_dataset=dataset_samsum_pt["validation"]
)

In [26]:
trainer.train()

Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=52, training_loss=192.06607554509088, metrics={'train_runtime': 984.8915, 'train_samples_per_second': 0.832, 'train_steps_per_second': 0.053, 'total_flos': 2366471355236352.0, 'train_loss': 192.06607554509088, 'epoch': 1.0})

In [27]:
#Evaluation

def generate_batch_sized_chunks(list_of_elements, batch_size):
    """A helper function to break a long list into chunks that we can process simultenously
    Yeild sucess batch size chunk from the list of elements"""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                               batch_size=16, device=device,
                                column_text="article",
                                column_summary="highlights"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)):

        inputs = tokenizer(article_batch, max_length=1024, truncation = True,
                           padding="max_length", return_tensors="pt")


        summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                          attention_mask = inputs["attention_mask"].to(device),
                          length_penalty=0.8, num_beams=8, max_length=128)
        """ parameter for length penalty ensures that the model does not generate sequences that are too long. """


    #finally we decode the generated text
    #Replace the token and add the generated text with the reference to the metric.

    decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                          clean_up_tokenization_spaces=True)
                         for s in summaries]

    decoded_summaries = [d.replace(" ", " ") for d in decoded_summaries]


    metric.add_batch(predictions=decoded_summaries, references=target_batch)

    #finally compute and return ROUGUE SCORE
    score = metric.compute()
    return score



In [28]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_metric = evaluate.load('rouge')

In [29]:
score = calculate_metric_on_test_ds(
    dataset_samsum['test'], rouge_metric, trainer.model, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
)

rouge_dict = dict((rn, score[rn]) for rn in rouge_names)
pd.DataFrame(rouge_dict, index = [f'pegasus'])

100%|██████████| 410/410 [1:04:50<00:00,  9.49s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.015748,0.0,0.015748,0.015748


In [30]:
#save model
model_pegasus.save_pretrained("pegasus-samsum-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [31]:
#save tokenizer
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

In [32]:
#loading
tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")

In [35]:
!pip install sentencepiece protobuf -q

In [46]:
# 1. Input ko Tokenize karein (Numbers mein badlein)
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding="max_length", max_length=1024).to(device)

# 2. Model se Summary Generate karwayein
# Hum wahi gen_kwargs use karenge jo aapne banaye thay
summary_ids = model_pegasus.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    length_penalty=0.8,
    num_beams=8,
    max_length=128
)

# 3. Generated IDs ko wapas Text mein badlein (Decode)
decoded_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Dialogue:")
print(sample_text)
print("\nModel Summary:")
print(decoded_summary)

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Model Summary:
day financial no need professionalreWere – downOct Food” News 5 him me well only because financial no need professionalreWere – SheOct Food” News 5 him me well only because financial no need professionalreWere – SheOct attend” News 5 him me well only because financial no need professionalreWere – SheOct attend” News 5 him me well only because financial no need professionalreWere – SheOct attend” News 5 him me well only because financial no need professionalreWere – SheOct attend” News 5 me well only because financial no need professionalreWere – SheOct Food”
